In [1]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
import evaluate
from peft import get_peft_model, LoraConfig, TaskType

c:\Users\lewy7\Documents\GitHub\roberta-hypernet-custom\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Load tokenizer and model
model_name = "xlm-roberta-base"
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)
base_model = XLMRobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1
)
model = get_peft_model(base_model, peft_config)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# Load CoLA dataset
dataset = load_dataset("glue", "cola")

In [4]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length")

encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [5]:
metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [6]:
training_args = TrainingArguments(
    output_dir="./results_lora",
    eval_strategy="steps",
    eval_steps=300,
    save_strategy="steps",
    save_steps=600,
    learning_rate=2e-5,
    per_device_train_batch_size=20,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs_lora",
    load_best_model_at_end=False,
    metric_for_best_model="matthews_correlation",
    dataloader_num_workers=4
)

In [7]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

C:\Users\lewy7\AppData\Local\Temp\ipykernel_16160\599301337.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [8]:
trainer.train()

Step,Training Loss,Validation Loss,Matthews Correlation
300,No log,0.618472,0.000000
600,0.620700,0.616111,0.000000
900,0.620700,0.613907,0.000000
1200,0.610200,0.616236,0.000000
1500,0.602900,0.608054,0.000000
1800,0.602900,0.611622,0.000000
2100,0.595600,0.607269,0.000000


TrainOutput(global_step=2140, training_loss=0.6063815571437372, metrics={'train_runtime': 2198.7115, 'train_samples_per_second': 19.445, 'train_steps_per_second': 0.973, 'total_flos': 1.136582024865792e+16, 'train_loss': 0.6063815571437372, 'epoch': 5.0})